In [ ]:
from matplotlib import pyplot as plt # plotting
import math
import sklearn
import optuna
#import GPy

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process import kernels
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ExpSineSquared, ConstantKernel, DotProduct


import scipy
from tqdm import tqdm
import torch
import numpy as np
import gymnasium as gym
import gym_unbalanced_disk, time
import torch.nn as nn
import inspect
import types

import stable_baselines3 as stable
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import LogEveryNTimesteps
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.sb2_compat.rmsprop_tf_like import RMSpropTFLike
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

from gymnasium.wrappers import TimeLimit

np.random.seed(101)


In [ ]:
unwrap = lambda theta: \
    -np.cos(theta)

custom_reward = lambda self: \
    unwrap(self.th)*(unwrap(self.th)>0) + ((self.omega/40)**2)*unwrap(self.th)/10


env = gym_unbalanced_disk.UnbalancedDisk(dt=0.025, umax=3.) #alternative
env.reward_fun = custom_reward

print(inspect.getfile(env.step))
print(inspect.getfile(env.reset))


In [ ]:
train_callback = LossRecorderCallback()

def objective(trial):

    # 2. Suggest values of the hyperparameters using a trial object.
    num_actor_layers = trial.suggest_int("num_actor_layers", 1,4)
    num_critic_layers = trial.suggest_int("num_critic_layers", 1,4)

    an_layers = []
    cn_layers = []

    for i in range(num_actor_layers): an_layers.append( trial.suggest_int(f'actor_layer{i}_size', 16, 128))
    for i in range(num_critic_layers): cn_layers.append( trial.suggest_int(f'critic_layer{i}_size', 16, 128))

    lr1 = trial.suggest_float("lr1", 0.0001, 0.01)
    lr3 = trial.suggest_float("lr3", 0.0001, 0.01)

    alpha1 = trial.suggest_float("alpha_critic", 0.01, 2)
    alpha2 = trial.suggest_float("alpha_entropy", 0.01, 2)

    learning_rate= lambda time: lr1*(0.99998**time) + lr3
    n_steps=200
    gamma=0.98
    ent_coef=alpha2
    vf_coef=alpha1

    policy = "MlpPolicy"

    policy_kwargs = dict(activation_fn=torch.nn.Sigmoid,
                     #features_extractor_class=CustomFeatures,
                     net_arch=dict(pi=an_layers, vf=cn_layers),
                     optimizer_class=RMSpropTFLike, 
                     optimizer_kwargs=dict(eps=1e-5)
                     )

    actor_crit = stable.A2C(policy, env, learning_rate=learning_rate, n_steps=n_steps, gamma=gamma, ent_coef=ent_coef, vf_coef=vf_coef, \
    max_grad_norm=0.5, use_rms_prop=False, normalize_advantage=True, stats_window_size=100, \
    tensorboard_log=None, policy_kwargs=policy_kwargs, verbose=1, _init_setup_model=True, use_sde=False)

    actor_crit.learn(total_timesteps=50_000, progress_bar=True, log_interval=100_000)

    mean_reward, std_reward = evaluate_policy(actor_crit, actor_crit.get_env(), n_eval_episodes=1)

    return mean_reward

# 3. Create a study object and optimize the objective function.
study = optuna.create_study(direction='maximize')





#actor_crit = stable.PPO(policy, env, learning_rate=learning_rate, n_steps=n_steps, gamma=gamma, ent_coef=ent_coef, vf_coef=vf_coef, \
#    max_grad_norm=0.5, normalize_advantage=True, stats_window_size=100, \
#    tensorboard_log=None, policy_kwargs=policy_kwargs, verbose=1, _init_setup_model=True)

#print(actor_crit.policy)

In [ ]:
study.optimize(objective, n_trials=10)

print(study.best_params)

In [4]:
import numpy as np
if np.array([1,2,3]).all()>0:
    print("piss")

piss


In [ ]:
num_actor_layers = 2
num_critic_layers = 3
an_layers = []
cn_layers = []

an_layers.append(67)
an_layers.append(61)

cn_layers.append(77)
cn_layers.append(18)
cn_layers.append(75)
lr1 = 0.007150645414764451
lr3 = 0.007982829376241094

alpha1 = 0.7975616158717067
alpha2 = 0.027445649596788302

learning_rate= lambda time: lr1*(0.99998**time) + lr3
n_steps=200
gamma=0.98
ent_coef=alpha2
vf_coef=alpha1
policy = "MlpPolicy"

policy_kwargs = dict(activation_fn=torch.nn.Sigmoid,
                 #features_extractor_class=CustomFeatures,
                 net_arch=dict(pi=an_layers, vf=cn_layers),
                 optimizer_class=RMSpropTFLike, 
                 optimizer_kwargs=dict(eps=1e-5)
                 )

actor_crit = stable.A2C(policy, env, learning_rate=learning_rate, n_steps=n_steps, gamma=gamma, ent_coef=ent_coef, vf_coef=vf_coef, \
    max_grad_norm=0.5, use_rms_prop=False, normalize_advantage=True, stats_window_size=100, \
    tensorboard_log=None, policy_kwargs=policy_kwargs, verbose=1, _init_setup_model=True, use_sde=False)

actor_crit.learn(total_timesteps=100_000, progress_bar=True, log_interval=100_000)


actor_crit.save("current_model")

vec_env = actor_crit.get_env()
obs = vec_env.reset()

for i in range(500):
    action, _state = actor_crit.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")
    # VecEnv resets automatically
    # if done:
    #   obs = vec_env.reset()

vec_env.close()

plt.figure(figsize=(10, 6))
plt.plot((train_callback.losses), label='Total Loss')
plt.plot((train_callback.policy_losses), label='Policy Loss')
plt.plot((train_callback.value_losses), label='Value Loss')
plt.plot((train_callback.entropy_losses), label='Entropy Loss')
plt.plot((train_callback.log_std), label='Sigma')

plt.xlabel('Update Step')
plt.ylabel('Loss Value')
plt.legend()
plt.title('Training Losses Over Time')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot((train_callback.th), label='theta')
plt.plot((train_callback.omega), label='omega')

plt.xlabel('Update Step')
plt.ylabel('Value')
plt.legend()
plt.title('Training state Over Time')
plt.show()